# Week 9 — The AgentOps release gate in CI

Notebook 05 produced thresholds. Notebook 06 produced evidence. Neither
*blocked* anything: a threshold that lives in a notebook is a belief, and a
belief does not stop a merge.

This notebook closes that gap with the
[AgentOps Accelerator](https://azure.github.io/agentops/) — an open-source CLI
that runs an evaluation dataset against an agent, scores it, fails on an exit
code, and writes a release evidence pack. It is the machinery behind
`agentops eval run`, `agentops doctor`, and the GitHub Actions workflows it
generates for you.

The most important lesson in this notebook is not how to run it. It is what to
do when the pipeline it generates conflicts with your platform's security
rules — because in this repository, it does.

Everything here runs offline. No cell makes a network request.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

## First: install the right package

Two unrelated products are called AgentOps.

| | Azure AgentOps Accelerator | AgentOps (agentops.ai) |
|---|---|---|
| PyPI | `agentops-accelerator` | `agentops` |
| Docs | azure.github.io/agentops | docs.agentops.ai |
| What it is | CLI for eval gates, Doctor, evidence packs | third-party SaaS observability |

`pip install agentops` installs the wrong one, and the mistake is quiet — you
get a working package with a similar vocabulary. The install line below is
taken from the accelerator's own generated workflow:

```bash
uv pip install "agentops-accelerator[foundry,agent]"
```

Claims you may find elsewhere about AgentOps automatically instrumenting
LangGraph refer to the other product. Verify the source before you act on a
search result here; the two documentation sets are easy to interleave.

## What a gate can point at

The accelerator infers the target kind from the *shape* of the `agent:` field
in `agentops.yaml`. There is no `type:` to set:

| `agent:` value | target kind |
|---|---|
| `"travel-agent:1"` | Foundry prompt agent, `name:version` |
| `"https://<foundry-project-host>/.../agents/<id>"` | Foundry hosted agent |
| `"https://api.example.com/chat"` | any HTTP/JSON agent |
| `"model:gpt-4o-mini"` | a raw model deployment |

The third row is the one that matters for framework choice. A LangGraph graph,
a Microsoft Agent Framework workflow, a Semantic Kernel app, and a hand-written
FastAPI route are **indistinguishable to the gate** as long as they serve the
same HTTP contract. The accelerator ships no framework adapter, and that is
the design: hosting, framework, and evaluation are three independent choices.

Notebook 04 built the graph shape and the target mapping. The next cell turns
that mapping into the configuration a gate actually consumes.

In [ ]:
def http_target_config(url, *, auth_env, captures_context, records_tool_calls):
    """Build the agentops.yaml fragment for a self-hosted agent endpoint."""

    if not url.startswith("https://"):
        raise ValueError("the gate must reach the agent over TLS")
    if auth_env.startswith(("Bearer ", "sk-", "https://")):
        # auth_header_env names an environment variable read at run time.
        # A literal here would be a secret committed to Git.
        raise ValueError("auth_header_env must name a variable, not hold a token")

    config = {
        "version": 1,
        "agent": url,
        "protocol": "http-json",
        "request_field": "message",
        "response_field": "text",
        "response_mode": "json",
        "auth_header_env": auth_env,
        "dataset": ".agentops/data/curriculum_cases.jsonl",
    }
    if records_tool_calls:
        config["tool_calls_field"] = "tool_calls"
    if captures_context:
        config["response_fields"] = {"context": "$response.context"}
    return config


def gate_measures(config):
    """Report what this configuration can and cannot detect."""

    measures = ["final answer quality", "latency"]
    if "tool_calls_field" in config:
        measures.append("tool trajectory")
    if "response_fields" in config:
        measures.append("groundedness")
    return measures


final_answer_only = http_target_config(
    "https://your-agent-host.example.com/chat",
    auth_env="APP_API_TOKEN",
    captures_context=False,
    records_tool_calls=False,
)
full_gate = http_target_config(
    "https://your-agent-host.example.com/chat",
    auth_env="APP_API_TOKEN",
    captures_context=True,
    records_tool_calls=True,
)

assert "tool trajectory" not in gate_measures(final_answer_only)
assert "tool trajectory" in gate_measures(full_gate)
assert "groundedness" in gate_measures(full_gate)

{
    "minimal": gate_measures(final_answer_only),
    "full": gate_measures(full_gate),
    "config": full_gate,
}

### What you just saw

Two configurations for the same endpoint, and the difference is two optional
fields.

`final_answer_only` is the default anyone reaches by accident, and it has a
specific blind spot: **an agent can reach the correct answer through a
forbidden tool and pass.** Notebook 04 built `TOOL_POLICY` precisely to stop
that call — but without `tool_calls_field`, the gate never observes whether
the policy held. The control exists and goes unmeasured.

`response_fields` has the same character for retrieval. Groundedness is not
something you switch on; it is something you earn by capturing the retrieved
text alongside the answer, which is the notebook-03 lesson arriving in CI.

Two failure modes worth recognising before you meet them:

- **A JSON parse error from a healthy endpoint** almost always means the
  endpoint streams. Set `response_mode: sse` (or `text`).
- **An empty `auth_header_env` variable** is a hard error, not a silent
  anonymous call. The guard above enforces the other half of that rule — the
  field names a variable and never holds the token.

### Change this and re-run

Pass `auth_env="Bearer abc123"` and watch `http_target_config` refuse. That
refusal is the whole no-secrets-in-Git rule expressed as three lines of code,
placed where the mistake is actually made.

In [ ]:
def gate_decision(exit_code):
    """Map an `agentops eval run` exit code onto an owner and an action."""

    return {
        0: ("pass", "none", "proceed"),
        2: ("fail", "the team that changed the agent", "do not merge"),
        1: ("error", "the team that owns the pipeline", "fix and re-run"),
    }[exit_code]


assert gate_decision(0)[0] == "pass"
assert gate_decision(2)[1] != gate_decision(1)[1]
assert gate_decision(2)[2] == "do not merge"

# A gate that collapses 2 and 1 into one red teaches people to re-run it.
{code: gate_decision(code) for code in (0, 2, 1)}

### What you just saw

Three outcomes, two of them red, and the two reds have **different owners**.

- `2` means the run worked and the agent did not clear the bar. That is a
  quality regression, owned by whoever changed the agent, prompt, tools, index,
  or model.
- `1` means the run itself broke — bad configuration, missing credentials, an
  unreachable endpoint. That is a pipeline incident, owned by a different team,
  and it says nothing at all about the agent.

Collapsing them into a single failure code is the most common way a gate loses
its authority. When every red might be infrastructure, re-running until green
becomes the reasonable response, and the gate stops being a gate.

### Change this and re-run

Collapse `2` and `1` onto the same tuple in `gate_decision`. The second
assertion fails, and that assertion is doing real work: it holds the two reds
apart. In CI the same collapse is invisible — the job simply goes red, someone
re-runs it, and the retry succeeds often enough to become a habit.

The local loop, before CI is involved at all:

```bash
agentops init             # bootstrap the .agentops/ workspace
agentops eval analyze     # what evaluators does this dataset shape earn?
agentops eval init        # write the recommended assets
agentops eval run         # exit 0 / 2 / 1
agentops doctor --evidence-pack
```

Get a green `eval run` locally before generating any workflow. A generated
pipeline around a gate that has never passed is a pipeline you cannot debug.

## What `agentops workflow generate` produces

```bash
agentops workflow analyze                  # inspect the repo, recommend wiring
agentops workflow generate --kinds pr      # start with the PR gate alone
agentops workflow generate --kinds pr,dev,qa,prod --deploy-mode auto --force
```

`--deploy-mode` is `auto` | `azd` | `prompt-agent` | `placeholder`.
`--doctor-gate` is `critical` (default) | `warning` | `none`.

The output is four workflows on a GitFlow model — `agentops-pr.yml`,
`agentops-deploy-dev.yml`, `agentops-deploy-qa.yml`, `agentops-deploy-prod.yml`,
plus `agentops-watchdog.yml`. The PR gate opens like this:

```yaml
on:
  pull_request:
    branches: [develop, "release/**", main]

permissions:
  contents: read
  pull-requests: write
  id-token: write

jobs:
  eval:
    runs-on: ubuntu-latest
    environment: dev
    steps:
      - uses: actions/checkout@v6
      - uses: azure/login@v3
        with:
          client-id: ${{ vars.AZURE_CLIENT_ID }}
          tenant-id: ${{ vars.AZURE_TENANT_ID }}
          subscription-id: ${{ vars.AZURE_SUBSCRIPTION_ID }}
```

It also posts the rendered report as an idempotent PR comment, uploads
`results.json` / `report.md` / the evidence pack as artifacts, and writes a job
summary. The configuration it needs is **repository variables, not secrets** —
`AZURE_CLIENT_ID`, `AZURE_TENANT_ID`, `AZURE_SUBSCRIPTION_ID`,
`AZURE_AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT` — with the CI
principal holding **Foundry User** and **Cognitive Services OpenAI User**.

That is a well-built pipeline. Read the next cell before you commit it here.

In [ ]:
PLATFORM_RULES = {
    "credential-free-pull-requests": (
        "no id-token: write or azure/login on a pull_request trigger"
    ),
    "no-environment-on-credentialed-jobs": (
        "the federated credential subject is a branch ref, not an environment"
    ),
    "actions-pinned-to-commit-sha": "third-party code runs beside a live identity",
}

generated_pr_workflow = {
    "trigger": "pull_request",
    "permissions": {"contents": "read", "pull-requests": "write", "id-token": "write"},
    "environment": "dev",
    "action_refs": ["actions/checkout@v6", "azure/login@v3"],
}


def violations(workflow):
    """Check a generated workflow against this platform's security boundary."""

    found = []
    credentialed = workflow["permissions"].get("id-token") == "write" or any(
        ref.startswith("azure/login") for ref in workflow["action_refs"]
    )
    if credentialed and workflow["trigger"] == "pull_request":
        found.append("credential-free-pull-requests")
    if credentialed and workflow.get("environment"):
        found.append("no-environment-on-credentialed-jobs")
    for ref in workflow["action_refs"]:
        if "@" in ref and len(ref.split("@")[1]) != 40:
            found.append("actions-pinned-to-commit-sha")
            break
    return found


found = violations(generated_pr_workflow)
assert set(found) == set(PLATFORM_RULES)

# The reconciliation: keep the gate, move the credential off the PR trigger.
reconciled = {
    "trigger": "push",
    "permissions": {"contents": "read", "id-token": "write"},
    "environment": None,
    "action_refs": ["actions/checkout@" + "a" * 40, "azure/login@" + "b" * 40],
}
assert violations(reconciled) == []

{
    "generated": [PLATFORM_RULES[name] for name in found],
    "reconciled": violations(reconciled),
}

### What you just saw

The workflow the accelerator generates breaks all three of this platform's
CI rules at once. It is not a badly written workflow — it is a correct
workflow for a different threat model, and this is the single most useful
thing in the notebook.

| generated `agentops-pr.yml` | this platform's rule | reconciliation |
|---|---|---|
| `id-token: write` on `pull_request` | PRs stay credential-free | move the credentialed gate to a post-merge `push`, or run only offline evaluators on PRs |
| `environment: dev` on a credentialed job | the FIC subject is a branch ref | drop `environment:`, or have the platform owner mint a matching environment credential first |
| `azure/login@v3` | pin every action to a full commit SHA | re-pin before committing |

Note that `violations` flags the pinning rule for **both** action
references, not only `azure/login`. `actions/checkout@v6` is a mutable tag too,
and it runs first — in the same job, next to the same identity.

The reason the first rule exists: a `pull_request` workflow runs code from the
proposed change. Handing it a live cloud identity means any contributor who can
open a pull request can run code as your CI principal. The accelerator's design
assumes a repository where that is acceptable. This one does not.

**So: do not commit `.github/workflows/agentops-*.yml` into this repository as
generated.** A downstream project not under these rules can, and should.

The reconciled variant keeps everything valuable — the same gate, the same
exit codes, the same evidence pack — and moves the credential to where a
branch-ref federated credential actually applies.

### Change this and re-run

Set `permissions["id-token"]` to `"read"` but leave `azure/login` in
`action_refs`. `violations` still reports both credential findings, because
`azure/login` is itself a credentialed step. A permissions block is a claim
about intent; the steps are the fact.

### The general lesson

A generated pipeline is a **proposal**. The accelerator cannot know your
federated-credential subject, your branch protection, or which triggers your
platform treats as untrusted. Reading generated CI before committing it is not
caution, it is the job.

In [ ]:
GATE_ARTIFACTS = {
    ".agentops/results/latest/results.json": "scores, machine-readable",
    ".agentops/results/latest/report.md": "the same run, for a human",
    ".agentops/agent/report.md": "Doctor findings",
    ".agentops/release/latest/evidence.json": "readiness summary, machine-readable",
    ".agentops/release/latest/evidence.md": "readiness summary, for a human",
}


def release_is_gated(evaluated_digest, deployed_digest, exit_code, doctor_critical):
    """The version deployed must be the exact version evaluated."""

    if evaluated_digest != deployed_digest:
        return False, "deployed artifact is not the evaluated artifact"
    if exit_code != 0:
        return False, f"eval gate returned {exit_code}"
    if doctor_critical:
        return False, "Doctor reported a critical finding"
    return True, "gated"


evaluated = "prompt:9f2c1a|git:4b7e0d"
rebuilt = "prompt:9f2c1a|git:aaaaaa"  # same prompt, different commit

assert release_is_gated(evaluated, evaluated, 0, False)[0]
assert not release_is_gated(evaluated, rebuilt, 0, False)[0]
assert not release_is_gated(evaluated, evaluated, 2, False)[0]
assert not release_is_gated(evaluated, evaluated, 0, True)[0]

{
    "artifacts": GATE_ARTIFACTS,
    "same_prompt_new_commit": release_is_gated(evaluated, rebuilt, 0, False),
}

### What you just saw

Four gate checks, and the second one is the one teams discover late.

`prompt:9f2c1a|git:aaaaaa` has an **identical prompt** and a different commit,
and it is refused. That is deliberate: an agent's behaviour is determined by
its prompt *and* its code, tools, index, and model. A version number alone
cannot identify what was evaluated, which is why the accelerator identifies a
candidate by prompt SHA plus git SHA.

The rule this enforces — *the version deployed is the exact version that was
evaluated* — sounds obvious and is violated constantly, usually by a prompt
edited in a portal after the gate ran.

Notebook 06's `release_manifest` records what was released. The evidence pack
records why it was allowed to be. A reviewer six months from now needs both.

### Change this and re-run

Make `release_is_gated` ignore `doctor_critical`. Three of the four assertions
still pass, and the gate now ships releases with known critical findings while
reporting success. Every check you remove makes the gate faster and quieter;
that is what makes removing them attractive.

## Exit criteria

Demonstrate, offline:

1. an `agentops.yaml` HTTP target that measures trajectory and groundedness,
   and the same target degraded to final-answer-only;
2. the three exit codes mapped to distinct owners and actions;
3. the three platform-rule violations in the generated PR workflow, and a
   reconciled workflow shape that clears all three;
4. a gate refusal caused by a prompt/commit digest mismatch.

Then state, in one sentence, which of your own agent's controls are currently
*enforced but unmeasured* — the controls that hold at run time and that no gate
would notice failing.

## Where this goes next

- Adopt the reconciled workflow shape in a project that is not this repository,
  and diff it against `agentops workflow generate` output before each upgrade.
- Feed `agentops eval promote-traces` output back into the dataset from
  notebook 06, review-first.
- Re-run `agentops eval analyze` whenever the dataset shape changes; the
  evaluators it earns are the gate's real scope.